# Testy integracyjne

Ten notebook opisuje testy integracyjne z katalogu `tests/integration/`: które elementy systemu współpracują, co dokładnie sprawdza każdy test
(stan początkowy, dane wejściowe, kroki, oczekiwany wynik, miejsce obserwacji), jaki jest wynik uruchomienia oraz jakie błędy znaleźliśmy.

Testy są automatyczne (pytest). Używają prawdziwych plików w tymczasowym katalogu notatek (`vault`), w aplikacji FastAPI
(`TestClient` albo prawdziwy serwer `uvicorn` na wolnym porcie) i, dla MCP po stdio, używa podprocesu `python -m mcp_server`.
Nic nie jest atrapą poza motywami wtyczek (jedna testowa wtyczka `Upper`).

### Jak czytać scenariusze
Każdy test ma pola:
- **Rodzaj**: *happy path* albo *błąd / złe dane* (sprawdzamy, jak elementy radzą sobie z niepoprawnymi danymi),
- **Stan początkowy**: co istnieje w vaulcie i w konfiguracji przed testem,
- **Dane wejściowe**, **Kroki**, **Oczekiwany wynik**,
- **Obserwujemy**: gdzie widać skutek integracji (plik na dysku, `.share-tokens.json`, kod HTTP, JSON-RPC, odpowiedź innego endpointu).

Test oznaczony **BUG** to znany błąd aplikacji. Test zawiera asercje oczekiwanego zachowania i jest oznaczony `xfail(strict=True)`,
więc w wynikach widać **XFAIL**.

Uruchomienie całości: `python -m pytest tests/integration -rx` (z katalogu głównego repozytorium).

## Testowane integracje

| Oznaczenie | Współpracujące elementy | Sposób komunikacji |
|---|---|---|
| A1–A3, A7 | REST API ↔ system plików (notatki, foldery, media, ZIP) | HTTP (`TestClient`) → zapis/odczyt plików |
| A1, A4 | REST API ↔ indeks notatek (wyszukiwanie, tagi, backlinki, graf) | wywołania funkcji w tym samym procesie po każdym zapisie |
| A2, A6 | REST API ↔ magazyn tokenów udostępniania (`.share-tokens.json`) ↔ publiczna strona | HTTP + plik JSON w vaulcie |
| A5 | REST API ↔ folder szablonów `_templates` | HTTP + odczyt plików |
| A8, M3 | endpointy ↔ uwierzytelnianie (sesja, klucz API) | HTTP, nagłówki, cookie |
| A9 | REST API ↔ menedżer wtyczek ↔ zapis notatki | hook `on_note_save` |
| M1–M2 | klient JSON-RPC ↔ `/mcp` ↔ serwer MCP | HTTP POST (JSON-RPC), prawdziwy `uvicorn` |
| M4–M5 | narzędzie MCP ↔ klient HTTP ↔ REST API ↔ dysk / indeks | wywołanie zwrotne (loopback) z kluczem API |
| M6 | proces `python -m mcp_server` ↔ działający backend | JSON-RPC po stdin/stdout + HTTP do backendu |

Wspólne przygotowanie (`tests/conftest.py`): tymczasowy vault, czyszczenie cache i indeksu przed każdym testem, klienci (`client`, `auth_client`, `logged_in_client`),
fixture `live_server` (prawdziwy `uvicorn` w wątku) i pomocnik `mcp_call`.

# Część A: REST API

## A1. Notatki

**Plik:** `tests/integration/api/test_api_notes.py`

**Integracja:** endpoint `/api/notes/*` ↔ zapis/odczyt plików `.md` w vaulcie ↔ indeks backlinków.

### A1.1 `test_note_lifecycle`

- **Rodzaj:** happy path
- **Stan początkowy:** Pusty vault.
- **Dane wejściowe:** Notatki `a/Note` (treść `# Hi\n[[Other]]`) i `Other.md`; kolejne treści `x`, `y`, `more`, `t`.
- **Kroki:** 1. `POST /api/notes/a/Note` (bez `.md`). 2. `POST /api/notes/Other.md`, potem `GET` (z i bez `include_backlinks`). 3. `POST` na istniejącą notatkę. 4. `PATCH` (dopisanie), `PATCH` z `add_timestamp`. 5. `DELETE`, ponowny `GET` i `DELETE`.
- **Oczekiwany wynik:** Plik powstaje jako `a/Note.md` (dopisane `.md`, utworzony folder). `GET` zwraca treść i backlink `a/Note.md`; `include_backlinks=false` go pomija. Ponowny `POST` to „saved” (nie „created”). `PATCH` dopisuje `\n\n` + treść, z timestampem dodaje separator `---` i datę. Po `DELETE` plik znika, `GET` i `DELETE` → 404.
- **Obserwujemy:** treść pliku na dysku, kody HTTP, pole `backlinks` w JSON, komunikaty `message`.

### A1.2 `test_patch_errors`

- **Rodzaj:** błąd / złe dane
- **Stan początkowy:** Notatka `n.md` o treści `x`.
- **Dane wejściowe:** `PATCH` z pustą treścią; `PATCH` na nieistniejącą notatkę.
- **Kroki:** 1. `PATCH /api/notes/n.md` z `content=""`. 2. `PATCH /api/notes/missing.md`.
- **Oczekiwany wynik:** Pierwsze → 400, drugie → 404.
- **Obserwujemy:** kody HTTP.

### A1.3 `test_name_clash_and_hash`

- **Rodzaj:** błąd / złe dane
- **Stan początkowy:** Notatka `Note.md`.
- **Dane wejściowe:** Ścieżki `note.md` (inna wielkość liter), `Note.md`, `a%23b.md` (znak `#`).
- **Kroki:** 1. `POST /api/notes/note.md`. 2. `POST /api/notes/Note.md`. 3. `POST /api/notes/a%23b.md`.
- **Oczekiwany wynik:** Kolizja nazwy bez rozróżniania wielkości liter → 409. Dokładnie ta sama nazwa → 200 (aktualizacja). `#` w nazwie → 400.
- **Obserwujemy:** kody HTTP.

### A1.4 `test_traversal_is_contained`

- **Rodzaj:** błąd / złe dane
- **Stan początkowy:** Plik `evil.md` **obok** vaultu o treści `keep`.
- **Dane wejściowe:** Ścieżka `..%2Fevil.md` (wyjście z vaultu).
- **Kroki:** 1. Próba `POST` poza vault. 2. `GET` i `DELETE` tej ścieżki.
- **Oczekiwany wynik:** `GET` i `DELETE` → 404. Plik `evil.md` poza vaultem pozostaje nietknięty. (Kod odpowiedzi na `POST` sprawdza osobny test poniżej.)
- **Obserwujemy:** kody HTTP, treść `evil.md` na dysku.

### A1.5 `test_traversal_write_is_rejected_as_client_error`

- **Rodzaj:** BUG (BR-1)
- **Stan początkowy:** Plik `evil.md` obok vaultu.
- **Dane wejściowe:** `POST /api/notes/..%2Fevil.md` z treścią `pwn`.
- **Kroki:** 1. Wysłać `POST` z próbą zapisu poza vault.
- **Oczekiwany wynik:** Odmowa jako błąd klienta: **400 lub 403**.
- **Obserwujemy:** kod HTTP. **Rzeczywisty: 500** (zapis jest blokowany, ale zgłaszany jako błąd serwera). Test oznaczony `xfail`.

### A1.6 `test_list_notes_pagination`

- **Rodzaj:** happy path
- **Stan początkowy:** Notatki `c.md`, `a.md`, `b.md`.
- **Dane wejściowe:** `GET /api/notes` oraz `?limit=2&offset=1`.
- **Kroki:** 1. Pobrać listę bez parametrów. 2. Pobrać z limitem i offsetem.
- **Oczekiwany wynik:** Bez parametrów: 3 notatki i brak pola `pagination`. Z parametrami: `b.md`, `c.md` (sortowanie po ścieżce), `total=3`.
- **Obserwujemy:** JSON odpowiedzi (`notes`, `pagination`).

In [ ]:
!cd ../.. && python -m pytest tests/integration/api/test_api_notes.py -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

: 

## A2. Foldery i przenoszenie

**Plik:** `tests/integration/api/test_api_folders.py`

**Integracja:** endpointy folderów ↔ system plików ↔ indeks notatek ↔ magazyn tokenów udostępniania.

### A2.1 `test_create_folder`

- **Rodzaj:** happy path + błąd
- **Stan początkowy:** Pusty vault.
- **Dane wejściowe:** Ścieżki `a/b`, `A`, `` (pusta), `x#y`.
- **Kroki:** 1. `POST /api/folders` z `a/b`. 2. Z `A` (inna wielkość liter niż `a`). 3. Z pustą ścieżką. 4. Z `x#y`.
- **Oczekiwany wynik:** `a/b` → 200 i katalog na dysku. `A` → 409 (kolizja). Pusta ścieżka i `#` → 400.
- **Obserwujemy:** katalog w vaulcie, kody HTTP.

### A2.2 `test_rename_and_move_folder_carry_notes`

- **Rodzaj:** happy path
- **Stan początkowy:** Notatka `old/n.md`, folder `dest/` z `keep.md`.
- **Dane wejściowe:** Zmiana nazwy `old` → `new`; przeniesienie `new` → `dest/new`.
- **Kroki:** 1. `POST /api/folders/rename`. 2. `GET /api/notes`. 3. `POST /api/folders/move`.
- **Oczekiwany wynik:** Notatka „jedzie” z folderem: istnieje `new/n.md`, nie ma `old/`, lista notatek pokazuje `new/n.md`, po przeniesieniu plik jest w `dest/new/n.md`.
- **Obserwujemy:** dysk, `GET /api/notes` (indeks).

### A2.3 `test_folder_move_errors`

- **Rodzaj:** błąd / złe dane
- **Stan początkowy:** Foldery `a/` i `b/`, każdy z `n.md`.
- **Dane wejściowe:** Brak `newPath`; `newPath` istniejący; zmiana nazwy nieistniejącego folderu.
- **Kroki:** 1. `move` z samym `oldPath`. 2. `move` `a` → `b`. 3. `rename` `nope` → `z`.
- **Oczekiwany wynik:** Wszystkie trzy → 400.
- **Obserwujemy:** kody HTTP.

### A2.4 `test_delete_non_empty_folder`

- **Rodzaj:** happy path
- **Stan początkowy:** Notatka `f/sub/n.md`.
- **Dane wejściowe:** `DELETE /api/folders/f`.
- **Kroki:** 1. Usunąć niepusty folder. 2. Sprawdzić dysk i `GET /api/notes`.
- **Oczekiwany wynik:** 200, folder znika razem z zawartością, lista notatek jest pusta (indeks zaktualizowany).
- **Obserwujemy:** dysk, `GET /api/notes`.

### A2.5 `test_delete_missing_folder_is_404`

- **Rodzaj:** BUG (BR-2)
- **Stan początkowy:** Pusty vault.
- **Dane wejściowe:** `DELETE /api/folders/nope`.
- **Kroki:** 1. Usunąć folder, którego nie ma.
- **Oczekiwany wynik:** **404** (nie ma zasobu).
- **Obserwujemy:** kod HTTP. **Rzeczywisty: 500**. Test oznaczony `xfail`.

### A2.6 `test_move_note`

- **Rodzaj:** happy path + błąd
- **Stan początkowy:** Notatka `n.md`.
- **Dane wejściowe:** Przeniesienie `n.md` → `d/n.md`; potem ponowne przeniesienie nieistniejącego `n.md`; brak `newPath`.
- **Kroki:** 1. `POST /api/notes/move`. 2. To samo dla już nieistniejącego źródła. 3. Żądanie bez `newPath`.
- **Oczekiwany wynik:** Pierwsze → 200 i plik w `d/n.md`. Drugie i trzecie → 400.
- **Obserwujemy:** dysk, kody HTTP.

### A2.7 `test_share_tokens_follow_moves`

- **Rodzaj:** happy path
- **Stan początkowy:** Notatki `a/n.md` i `ab/m.md`, obie udostępnione (tokeny `t_n`, `t_m`).
- **Dane wejściowe:** Zmiana nazwy folderu `a` → `c`; usunięcie folderu `ab`; przeniesienie `c/n.md` → `moved.md`.
- **Kroki:** 1. `rename` folderu. 2. `GET /api/shared-notes`, `GET /share/t_n`. 3. `DELETE /api/folders/ab`, `GET /share/t_m`. 4. `POST /api/notes/move`, `GET /api/shared-notes`.
- **Oczekiwany wynik:** Token „idzie za” notatką: `shared-notes` = `ab/m.md`, `c/n.md` (prefiks `a/` nie dotyka `ab/`), link `t_n` działa. Po usunięciu folderu `ab` link `t_m` → 404. Po przeniesieniu notatki lista = `moved.md`.
- **Obserwujemy:** `GET /api/shared-notes`, kody `GET /share/{token}`.

In [ ]:
!cd ../.. && python -m pytest tests/integration/api/test_api_folders.py -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

## A3. Media (obrazy)

**Plik:** `tests/integration/api/test_api_media.py`

**Integracja:** endpointy mediów ↔ walidacja typu/rozmiaru ↔ zapis plików w `_attachments` ↔ nazwy plików.

### A3.1 `test_upload_and_get`

- **Rodzaj:** happy path
- **Stan początkowy:** Pusty vault.
- **Dane wejściowe:** Plik `pic.png` (nagłówek PNG), `note_path=folder/n.md`; drugi upload bez ścieżki notatki.
- **Kroki:** 1. `POST /api/upload-media`. 2. `GET /api/media/{zwrócona ścieżka}`. 3. Upload dla notatki w katalogu głównym.
- **Oczekiwany wynik:** Odpowiedź ma `type=image`, ścieżka `folder/_attachments/pic-….png`, plik na dysku ma te same bajty, `GET` zwraca te same bajty. Dla notatki w korzeniu ścieżka zaczyna się od `_attachments/`.
- **Obserwujemy:** dysk, treść odpowiedzi `GET`.

### A3.2 `test_upload_rejections`

- **Rodzaj:** błąd / złe dane
- **Stan początkowy:** Pusty vault.
- **Dane wejściowe:** Plik `a.txt` (`text/plain`); poprawny PNG przy limicie rozmiaru ustawionym na 0 MB.
- **Kroki:** 1. Upload pliku tekstowego. 2. Ustawić `UPLOAD_MAX_IMAGE_MB=0` i wgrać PNG.
- **Oczekiwany wynik:** Oba → 400 (zły typ, zbyt duży plik).
- **Obserwujemy:** kody HTTP.

### A3.3 `test_upload_next_to_notes`

- **Rodzaj:** happy path + błąd
- **Stan początkowy:** Pusty vault.
- **Dane wejściowe:** Upload z `next_to_notes=1`, `content_folder=d` (PNG), potem JPEG.
- **Kroki:** 1. Upload PNG obok notatek. 2. Upload JPEG w tym samym trybie.
- **Oczekiwany wynik:** PNG → 200, nazwa `d/drawing-…`. JPEG → 400 (w tym trybie tylko PNG).
- **Obserwujemy:** pole `path`, kody HTTP.

### A3.4 `test_get_media_errors`

- **Rodzaj:** błąd / złe dane
- **Stan początkowy:** Notatka `n.md`.
- **Dane wejściowe:** `/api/media/n.md`, `/api/media/nope.png`, `/api/media/..%2Fx.png`.
- **Kroki:** 1. `GET` pliku, który nie jest mediem. 2. `GET` nieistniejącego. 3. `GET` z próbą wyjścia z vaultu.
- **Oczekiwany wynik:** 400, 404, oraz 403 lub 404.
- **Obserwujemy:** kody HTTP.

### A3.5 `test_put_media`

- **Rodzaj:** happy path + błąd
- **Stan początkowy:** Plik `drawing-1.png` (PNG) i `pic.png` w vaulcie.
- **Dane wejściowe:** Nowa treść PNG; treść nie-PNG; `drawing-2.png` (nie istnieje); `pic.png` (zła nazwa).
- **Kroki:** 1. `PUT` nowych bajtów PNG. 2. `PUT` bajtów nie-PNG. 3. `PUT` nieistniejącego pliku. 4. `PUT` pliku o nazwie spoza reguły `drawing-*`.
- **Oczekiwany wynik:** Pierwszy → 200 i plik nadpisany. Drugi → 400. Trzeci → 404. Czwarty → 400.
- **Obserwujemy:** dysk, kody HTTP.

### A3.6 `test_move_media`

- **Rodzaj:** happy path + błąd
- **Stan początkowy:** Pliki `a.png`, `b.png`, `n.md`.
- **Dane wejściowe:** Przeniesienia: `a.png`→`d/a.png`, `d/a.png`→`b.png`, `zz.png`→`y.png`, `b.png`→`b.txt`, `n.md`→`n2.md`, `b.png`→`../out.png`.
- **Kroki:** 1. Kolejne `POST /api/media/move`.
- **Oczekiwany wynik:** 200 i plik w `d/`; 409 (cel istnieje); 404 (brak źródła); 200 (sprawdzane jest tylko rozszerzenie źródła); 400 (źródło nie jest mediem); 403 (wyjście poza vault).
- **Obserwujemy:** dysk, kody HTTP.

In [ ]:
!cd ../.. && python -m pytest tests/integration/api/test_api_media.py -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

## A4. Wyszukiwanie, tagi, graf, statystyki

**Plik:** `tests/integration/api/test_api_search_graph.py`

**Integracja:** zapis notatek ↔ indeks (wyszukiwanie, tagi, linki/graf) ↔ statystyki: czy indeks nadąża za zmianami.

### A4.1 `test_search_follows_writes`

- **Rodzaj:** happy path
- **Stan początkowy:** Pusty vault.
- **Dane wejściowe:** Notatka `a.md` o treści „the zebra runs”, potem „the lion runs”.
- **Kroki:** 1. `POST` notatki, `GET /api/search?q=zebra` (i `ZEBRA`). 2. Nadpisanie treści. 3. Przeniesienie do `d/b.md`. 4. Usunięcie.
- **Oczekiwany wynik:** Wynik wyszukiwania stale zgodny z zawartością: `zebra` → `a.md` (wielkość liter bez znaczenia), po nadpisaniu `zebra` → brak, `lion` → `a.md`, po przeniesieniu `d/b.md`, po usunięciu brak wyników.
- **Obserwujemy:** `GET /api/search`.

### A4.2 `test_search_edge_cases`

- **Rodzaj:** błąd / złe dane
- **Stan początkowy:** Notatki `a.md`, `b.md`, `c.md` z słowem `needle`.
- **Dane wejściowe:** Zapytanie `"  "`; `needle` z `limit=2&offset=2`; wyłączone wyszukiwanie w konfiguracji.
- **Kroki:** 1. Zapytanie z samych spacji. 2. Zapytanie ze stronicowaniem. 3. Wyłączyć `search.enabled` i szukać.
- **Oczekiwany wynik:** Puste wyniki; jedna strona z `c.md` i `total=3`; po wyłączeniu → 403.
- **Obserwujemy:** JSON wyników, kod HTTP.

### A4.3 `test_tags_follow_writes`

- **Rodzaj:** happy path
- **Stan początkowy:** Notatki `a.md` (tagi `Alpha`, `beta`) i `b.md` (tag `alpha`).
- **Dane wejściowe:** Zapytania do `/api/tags`, `/api/tags/ALPHA`, `?limit=1`; nadpisanie `b.md` bez tagów; usunięcie `a.md`.
- **Kroki:** 1. `GET /api/tags`. 2. `GET /api/tags/ALPHA`. 3. `GET /api/tags/alpha?limit=1`. 4. `POST` bez tagów, `DELETE`.
- **Oczekiwany wynik:** Tagi liczone bez rozróżniania wielkości liter (`alpha: 2`, `beta: 1`). Po edycji `alpha: 1`, po usunięciu brak tagów. Nieistniejący tag → pusta lista.
- **Obserwujemy:** `GET /api/tags`, `GET /api/tags/{tag}`.

### A4.4 `test_graph_follows_renames_and_deletes`

- **Rodzaj:** happy path
- **Stan początkowy:** Notatki `a.md` (`[[b]]`) i `b.md`.
- **Dane wejściowe:** Przeniesienie `b.md` → `d/b.md`; usunięcie `d/b.md`.
- **Kroki:** 1. `GET /api/graph`. 2. Przenieść `b`. 3. `GET /api/graph`. 4. Usunąć `b`. 5. `GET /api/graph`.
- **Oczekiwany wynik:** Początkowo krawędź `a.md → b.md`; po przeniesieniu `a.md → d/b.md`; po usunięciu brak krawędzi.
- **Obserwujemy:** `GET /api/graph`.

### A4.5 `test_stats`

- **Rodzaj:** happy path
- **Stan początkowy:** Notatka `f/a.md` z tagiem `t` i szablon `_templates/T.md`.
- **Dane wejściowe:** `GET /api/stats`, `GET /api/index/stats`.
- **Kroki:** 1. Pobrać statystyki.
- **Oczekiwany wynik:** `notes_count=2`, `tags_count=1`, `templates_count=1`, `folders_count=2`; statystyki indeksu niepuste.
- **Obserwujemy:** JSON statystyk.

In [ ]:
!cd ../.. && python -m pytest tests/integration/api/test_api_search_graph.py -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

## A5. Szablony

**Plik:** `tests/integration/api/test_api_templates.py`

**Integracja:** endpointy szablonów ↔ folder `_templates` ↔ tworzenie notatki z podstawieniem zmiennych.

### A5.1 `test_list_and_get`

- **Rodzaj:** happy path + błąd
- **Stan początkowy:** `_templates/Daily.md`, `Other.md`, `ignore.txt`.
- **Dane wejściowe:** `GET /api/templates`, `GET /api/templates/Daily`, `GET /api/templates/Nope`.
- **Kroki:** 1. Lista szablonów. 2. Pobranie istniejącego. 3. Pobranie nieistniejącego.
- **Oczekiwany wynik:** Lista: `Daily`, `Other` (plik `.txt` pominięty). Treść `Daily` zaczyna się od `# {{title}}`. Brak szablonu → 404.
- **Obserwujemy:** JSON, kod HTTP.

### A5.2 `test_list_without_templates_folder`

- **Rodzaj:** happy path
- **Stan początkowy:** Vault bez folderu `_templates`.
- **Dane wejściowe:** `GET /api/templates`.
- **Kroki:** 1. Pobrać listę.
- **Oczekiwany wynik:** `{"templates": []}`, bez błędu.
- **Obserwujemy:** JSON.

### A5.3 `test_create_note_from_template`

- **Rodzaj:** happy path
- **Stan początkowy:** Szablon `Daily` z `{{title}}`, `{{folder}}`, `{{date}}`.
- **Dane wejściowe:** `POST /api/templates/create-note` z `Daily` i `proj/Plan.md`.
- **Kroki:** 1. Utworzyć notatkę z szablonu. 2. Odczytać plik.
- **Oczekiwany wynik:** Plik `proj/Plan.md` zaczyna się od `# Plan\nin proj on `, kończy datą `RRRR-MM-DD`, nie zawiera `{{`; odpowiedź zawiera tę samą treść.
- **Obserwujemy:** plik na dysku, pole `content`.

### A5.4 `test_create_note_errors`

- **Rodzaj:** błąd / złe dane
- **Stan początkowy:** Szablon `Daily` i notatka `exists.md`.
- **Dane wejściowe:** `notePath=exists.md`, `EXISTS.md`; brak `notePath`; szablon `Nope`.
- **Kroki:** 1. Utworzyć na istniejącej nazwie. 2. To samo inną wielkością liter. 3. Bez ścieżki. 4. Z nieistniejącym szablonem.
- **Oczekiwany wynik:** 409, 409, 400, 404.
- **Obserwujemy:** kody HTTP.

In [ ]:
!cd ../.. && python -m pytest tests/integration/api/test_api_templates.py -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

## A6. Udostępnianie notatek

**Plik:** `tests/integration/api/test_api_sharing.py`

**Integracja:** endpointy `/api/share*` ↔ magazyn tokenów (`.share-tokens.json` w vaulcie) ↔ publiczna strona `/share/{token}` ↔ notatki na dysku.

### A6.1 `test_create_and_view`

- **Rodzaj:** happy path
- **Stan początkowy:** Notatka `n.md` z frontmatterem (`title: secret meta`) i treścią `# Public body`.
- **Dane wejściowe:** `POST /api/share/n` (bez slugu).
- **Kroki:** 1. Udostępnić notatkę, ponownie (`n.md`). 2. Otworzyć `/share/{token}`. 3. `GET /api/share/n`, `GET /api/shared-notes`.
- **Oczekiwany wynik:** Zwrócony token i URL kończący się tokenem; ponowne udostępnienie zwraca ten sam token. Strona publiczna pokazuje „Public body” i **nie** pokazuje frontmattera. Status `shared=true`, lista = `[n.md]`.
- **Obserwujemy:** HTML strony, JSON statusu.

### A6.2 `test_custom_slug_and_rename`

- **Rodzaj:** happy path
- **Stan początkowy:** Notatka `n.md`.
- **Dane wejściowe:** Slugi `my-slug`, potem `new-slug`.
- **Kroki:** 1. Udostępnić ze slugiem `my-slug`. 2. Ponownie z `new-slug`. 3. Otworzyć oba URL-e.
- **Oczekiwany wynik:** Token = slug. Po zmianie stary URL → 404, nowy → 200.
- **Obserwujemy:** kody `GET /share/{slug}`.

### A6.3 `test_slug_errors`

- **Rodzaj:** błąd / złe dane
- **Stan początkowy:** Notatki `n.md`, `m.md`; slug `taken` należy do `n`.
- **Dane wejściowe:** Slugi: `taken` dla `m`, `ab`, 65 × `x`, `bad slug!`; udostępnienie nieistniejącej notatki.
- **Kroki:** 1. Udostępnić `m` ze slugiem zajętym. 2. Z za krótkim / za długim / z niedozwolonymi znakami. 3. Udostępnić `missing`.
- **Oczekiwany wynik:** 409 `taken`; 400 z powodami `too_short`, `too_long`, `invalid_chars`; nieistniejąca notatka → 404.
- **Obserwujemy:** kod HTTP i `detail.reason`.

### A6.4 `test_slug_check`

- **Rodzaj:** happy path + błąd
- **Stan początkowy:** Notatka `n.md` udostępniona jako `mine`.
- **Dane wejściowe:** `GET /api/share-slug` dla `free-one`, `mine`, `mine`+`note_path=n`, `a`.
- **Kroki:** 1. Sprawdzić cztery sluga.
- **Oczekiwany wynik:** `free-one` dostępny; `mine` zajęty (`taken`); `mine` dla własnej notatki dostępny; `a` → `too_short`.
- **Obserwujemy:** JSON `available`/`reason`.

### A6.5 `test_revoke`

- **Rodzaj:** happy path
- **Stan początkowy:** Udostępniona notatka `n.md`.
- **Dane wejściowe:** `DELETE /api/share/n`, powtórzone.
- **Kroki:** 1. Cofnąć udostępnienie. 2. `GET /share/{token}`. 3. Cofnąć ponownie. 4. `GET /api/share/n`.
- **Oczekiwany wynik:** Pierwsze cofnięcie `success=true`, link → 404; drugie `success=false`; status `{shared: false}`.
- **Obserwujemy:** JSON, kod `GET /share/{token}`.

### A6.6 `test_deleted_note_link_is_404`

- **Rodzaj:** błąd / złe dane
- **Stan początkowy:** Udostępniona notatka `n.md`.
- **Dane wejściowe:** Usunięcie pliku `n.md` bezpośrednio z dysku (poza aplikacją).
- **Kroki:** 1. Skasować plik na dysku. 2. `GET /share/{token}`. 3. `GET /api/shared-notes`.
- **Oczekiwany wynik:** Link → 404, lista udostępnionych pusta (aplikacja radzi sobie z „osieroconym” tokenem).
- **Obserwujemy:** kody HTTP, JSON.

### A6.7 `test_public_with_auth_on`

- **Rodzaj:** happy path + błąd
- **Stan początkowy:** Włączone uwierzytelnianie; notatka `n.md`.
- **Dane wejściowe:** Udostępnienie z kluczem API; odczyt linku bez logowania; `POST /api/share/n` bez klucza.
- **Kroki:** 1. `POST /api/share/n` z `X-API-Key`. 2. `GET /share/{token}` anonimowo. 3. `POST /api/share/n` anonimowo.
- **Oczekiwany wynik:** Strona publiczna dostępna bez logowania (200); tworzenie udostępnienia bez klucza → 401.
- **Obserwujemy:** kody HTTP.

### A6.8 `test_deleting_note_through_api_removes_token`

- **Rodzaj:** happy path
- **Stan początkowy:** Udostępniona notatka `n.md`.
- **Dane wejściowe:** `DELETE /api/notes/n.md`.
- **Kroki:** 1. Usunąć notatkę przez API. 2. Wczytać `.share-tokens.json`. 3. Sprawdzić `shared-notes` i `/share/{token}`.
- **Oczekiwany wynik:** Token usunięty z pliku od razu (nie „leniwie”), lista pusta, link → 404.
- **Obserwujemy:** `.share-tokens.json`, kody HTTP.

### A6.9 `test_theme_is_stored_and_used`

- **Rodzaj:** happy path
- **Stan początkowy:** Notatki `n.md`, `m.md`.
- **Dane wejściowe:** Udostępnienie z `theme=dark`; zmiana sluga; druga notatka bez motywu.
- **Kroki:** 1. Udostępnić z motywem ciemnym. 2. Odczytać status i plik tokenów. 3. Zmienić slug. 4. Udostępnić drugą notatkę domyślnie.
- **Oczekiwany wynik:** Motyw zapisany w statusie i w `.share-tokens.json`; zmiana sluga zachowuje motyw (identyczny HTML); domyślny motyw jest jasny (HTML inny).
- **Obserwujemy:** JSON, `.share-tokens.json`, HTML strony.

### A6.10 `test_public_origin_in_share_urls`

- **Rodzaj:** happy path
- **Stan początkowy:** Notatka `n.md`; ustawiony `SHARE_PUBLIC_ORIGIN=https://notes.example.com`.
- **Dane wejściowe:** `POST /api/share/n`, `GET /api/share/n`.
- **Kroki:** 1. Udostępnić. 2. Pobrać status.
- **Oczekiwany wynik:** URL = `https://notes.example.com/share/{token}` w obu odpowiedziach.
- **Obserwujemy:** pole `url`.

### A6.11 `test_tokens_are_persisted_on_disk`

- **Rodzaj:** happy path
- **Stan początkowy:** Notatki `n.md`, `m.md`.
- **Dane wejściowe:** Udostępnienie `n` z motywem `dark`; ręcznie podmieniony `.share-tokens.json` (token `abc123` → `m.md`).
- **Kroki:** 1. Udostępnić `n`, odczytać wpis w pliku. 2. Nadpisać plik tokenów „z backupu”. 3. Otworzyć `/share/abc123` i stary token.
- **Oczekiwany wynik:** Wpis ma `path`, `theme`, `created`. Token z podmienionego pliku działa, stary → 404, lista = `[m.md]` (aplikacja czyta plik tokenów na bieżąco).
- **Obserwujemy:** `.share-tokens.json`, kody HTTP.

In [ ]:
!cd ../.. && python -m pytest tests/integration/api/test_api_sharing.py -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

## A7. Eksport (HTML, ZIP)

**Plik:** `tests/integration/api/test_api_export.py`

**Integracja:** endpointy eksportu ↔ pliki notatek i obrazów ↔ szablony motywów ↔ archiwum ZIP.

### A7.1 `test_export_download_and_inline`

- **Rodzaj:** happy path
- **Stan początkowy:** Notatka `d/My Note.md` z frontmatterem i nagłówkiem.
- **Dane wejściowe:** `GET /api/export/d/My Note.md` oraz z `?download=false`.
- **Kroki:** 1. Pobrać eksport. 2. Pobrać wersję inline.
- **Oczekiwany wynik:** HTML z `Content-Disposition: attachment` i nazwą `My%20Note.html`; frontmatter usunięty; wersja inline bez nagłówka `Content-Disposition`.
- **Obserwujemy:** nagłówki HTTP, treść HTML.

### A7.2 `test_export_theme_and_fallback`

- **Rodzaj:** happy path + błąd
- **Stan początkowy:** Notatka `n.md`.
- **Dane wejściowe:** `?theme=dark`, `?theme=light`, `?theme=does-not-exist`; eksport `missing.md`.
- **Kroki:** 1. Wyeksportować w dwóch motywach. 2. Z nieznanym motywem. 3. Nieistniejącą notatkę.
- **Oczekiwany wynik:** Dark ≠ light; nieznany motyw → 200 (powrót do jasnego); brak notatki → 404.
- **Obserwujemy:** treść HTML, kody HTTP.

### A7.3 `test_export_embeds_images`

- **Rodzaj:** happy path
- **Stan początkowy:** Notatka `![pic](pic.png)` i plik `pic.png`.
- **Dane wejściowe:** `GET /api/export/n.md`.
- **Kroki:** 1. Wyeksportować.
- **Oczekiwany wynik:** Obraz wbudowany w HTML jako `data:image/png;base64,…`.
- **Obserwujemy:** treść HTML.

### A7.4 `test_archive_folder_and_vault`

- **Rodzaj:** happy path
- **Stan początkowy:** `f/a.md`, `f/sub/b.md`, `top.md`, `f/_attachments/p.png`.
- **Dane wejściowe:** `GET /api/archive?folder=f` oraz bez parametru.
- **Kroki:** 1. Pobrać ZIP folderu. 2. Pobrać ZIP całego vaultu.
- **Oczekiwany wynik:** ZIP folderu: `_attachments/p.png`, `a.md`, `sub/b.md` (nazwa `f.zip`). ZIP vaultu: te same pliki z prefiksem `f/` plus `top.md`.
- **Obserwujemy:** lista plików w ZIP, `Content-Disposition`.

### A7.5 `test_archive_excludes_hidden`

- **Rodzaj:** happy path
- **Stan początkowy:** `f/a.md`, `f/.hidden.md`, `f/.git/config`, `.share-tokens.json`.
- **Dane wejściowe:** `GET /api/archive?folder=f` i bez parametru.
- **Kroki:** 1. Pobrać ZIP folderu. 2. Pobrać ZIP vaultu.
- **Oczekiwany wynik:** W ZIP tylko `a.md`; `.share-tokens.json` nie trafia do archiwum.
- **Obserwujemy:** lista plików w ZIP.

### A7.6 `test_archive_errors`

- **Rodzaj:** błąd / złe dane
- **Stan początkowy:** `a.md`, katalogi `.git` i `empty`.
- **Dane wejściowe:** `folder` = `nope`, `.git`, `..`, `a.md`, `empty`.
- **Kroki:** 1. Pobrać archiwum dla każdej z pięciu wartości.
- **Oczekiwany wynik:** Zawsze 404 (brak folderu, ukryty, wyjście poza vault, plik zamiast folderu, pusty folder).
- **Obserwujemy:** kody HTTP.

In [ ]:
!cd ../.. && python -m pytest tests/integration/api/test_api_export.py -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

## A8. Uwierzytelnianie

**Plik:** `tests/integration/api/test_api_auth.py`

**Integracja:** middleware/zależność `require_auth` ↔ sesja (cookie) ↔ klucz API ↔ endpointy chronione i publiczne.

### A8.1 `test_auth_off_is_open`

- **Rodzaj:** happy path
- **Stan początkowy:** Uwierzytelnianie wyłączone.
- **Dane wejściowe:** `GET /api/notes`, `/login`, `/api/config`.
- **Kroki:** 1. Wywołać trzy endpointy.
- **Oczekiwany wynik:** Dostęp otwarty (200); `/login` przekierowuje 303 na `/`; konfiguracja `authentication.enabled=false`.
- **Obserwujemy:** kody HTTP, `Location`.

### A8.2 `test_auth_on_rejects_anonymous`

- **Rodzaj:** błąd / złe dane
- **Stan początkowy:** Uwierzytelnianie włączone, brak logowania.
- **Dane wejściowe:** `/api/notes`, `/`, publiczne `/health`, `/login`, `/api/themes/light`, `/api/locales`, `/manifest.json`, chronione `/api/themes`.
- **Kroki:** 1. Wywołać każdy z endpointów anonimowo.
- **Oczekiwany wynik:** `/api/notes` → 401 JSON `Not authenticated`; `/` → 303 na `/login`; publiczne → 200; `/api/themes` (lista) → 401.
- **Obserwujemy:** kody HTTP, JSON.

### A8.3 `test_login`

- **Rodzaj:** happy path + błąd
- **Stan początkowy:** Uwierzytelnianie włączone.
- **Dane wejściowe:** Hasło `wrong` i poprawne.
- **Kroki:** 1. `POST /login` ze złym hasłem. 2. `GET /api/notes`. 3. `POST /login` z dobrym hasłem. 4. `GET /api/notes`, `GET /login`.
- **Oczekiwany wynik:** Złe hasło → redirect `/login?error=incorrect_password`, API nadal 401. Dobre → 303 na `/`, a potem API 200 (cookie sesji); `/login` przekierowuje na `/`.
- **Obserwujemy:** `Location`, kody HTTP.

### A8.4 `test_logout`

- **Rodzaj:** happy path
- **Stan początkowy:** Zalogowany klient (sesja).
- **Dane wejściowe:** `GET /logout`, `POST /logout`.
- **Kroki:** 1. `GET /logout`. 2. `GET /api/notes`. 3. `POST /logout`. 4. `GET /api/notes`.
- **Oczekiwany wynik:** `GET` → 405 i sesja trwa (200). `POST` → 303 na `/login`, potem API → 401.
- **Obserwujemy:** kody HTTP.

### A8.5 `test_api_key`

- **Rodzaj:** happy path + błąd
- **Stan początkowy:** Uwierzytelnianie włączone.
- **Dane wejściowe:** Nagłówki: `X-API-Key` poprawny / zły / pusty; `Authorization: Bearer` poprawny / zły.
- **Kroki:** 1. `GET /api/notes` z każdym wariantem (5 przypadków).
- **Oczekiwany wynik:** Poprawny klucz (obiema metodami) → 200, pozostałe → 401.
- **Obserwujemy:** kody HTTP.

In [ ]:
!cd ../.. && python -m pytest tests/integration/api/test_api_auth.py -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

## A9. System i wtyczki

**Plik:** `tests/integration/api/test_api_system_plugins.py`

**Integracja:** endpointy systemowe ↔ pliki motywów/tłumaczeń/manifestu ↔ menedżer wtyczek ↔ zapis notatki.

### A9.1 `test_system_endpoints`

- **Rodzaj:** happy path + błąd
- **Stan początkowy:** Domyślna konfiguracja, uwierzytelnianie wyłączone.
- **Dane wejściowe:** `/health`, `/api/config`, `/api/themes`, `/api/themes/{id}`, `/api/locales`, `/api/locales/{code}`, `/manifest.json`, `/sw.js`; złe id: `nope`, `bad.code`.
- **Kroki:** 1. Wywołać wszystkie endpointy. 2. Wywołać z nieistniejącym motywem i tłumaczeniem oraz niepoprawnym kodem.
- **Oczekiwany wynik:** Poprawne odpowiedzi (np. `healthy`, motyw `light`, tłumaczenie `en-US`, `sw.js` jako JavaScript). Nieznany motyw/tłumaczenie → 404, niepoprawny kod → 400.
- **Obserwujemy:** JSON, nagłówek `content-type`, kody HTTP.

### A9.2 `test_plugin_list_and_toggle`

- **Rodzaj:** happy path
- **Stan początkowy:** Zarejestrowana atrapa wtyczki `Upper` (plik konfiguracji w katalogu tymczasowym).
- **Dane wejściowe:** `GET /api/plugins`, `POST /api/plugins/upper/toggle` z `enabled=false`, potem `true`.
- **Kroki:** 1. Pobrać listę. 2. Wyłączyć. 3. Włączyć.
- **Oczekiwany wynik:** Wtyczka na liście; przełączenie zwraca `{success, plugin, enabled}` i zmienia stan obiektu wtyczki.
- **Obserwujemy:** JSON, atrybut `enabled` wtyczki.

### A9.3 `test_plugin_hook_transforms_content_only_while_enabled`

- **Rodzaj:** happy path
- **Stan początkowy:** Wtyczka `Upper` (hook `on_note_save` zamienia treść na wielkie litery).
- **Dane wejściowe:** Zapis notatek `a.md` i `b.md` o treści `abc`, między nimi wyłączenie wtyczki.
- **Kroki:** 1. Zapisać `a.md` przy włączonej wtyczce. 2. Wyłączyć wtyczkę. 3. Zapisać `b.md`.
- **Oczekiwany wynik:** `a.md` na dysku = `ABC`, `b.md` = `abc`.
- **Obserwujemy:** treść plików na dysku.

In [ ]:
!cd ../.. && python -m pytest tests/integration/api/test_api_system_plugins.py -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

# Część M: MCP

## M1. Transport HTTP `/mcp`

**Plik:** `tests/integration/mcp/test_mcp_transport.py`

**Integracja:** klient HTTP (JSON-RPC) ↔ router `/mcp` (`backend/mcp_http.py`) ↔ serwer MCP; część testów na żywym serwerze uvicorn.

### M1.1 `test_ping_ok`

- **Rodzaj:** happy path
- **Stan początkowy:** Działający serwer na wolnym porcie.
- **Dane wejściowe:** `POST /mcp` z `ping`.
- **Kroki:** 1. Wysłać żądanie.
- **Oczekiwany wynik:** 200 i `{"jsonrpc":"2.0","id":1,"result":{}}`.
- **Obserwujemy:** treść odpowiedzi.

### M1.2 `test_wildcard_origin_allowed`

- **Rodzaj:** happy path
- **Stan początkowy:** Domyślna konfiguracja (dozwolone wszystkie źródła).
- **Dane wejściowe:** `ping` z nagłówkiem `Origin: http://evil.example`.
- **Kroki:** 1. Wysłać żądanie z obcym `Origin`.
- **Oczekiwany wynik:** 200 (domyślnie dozwolone `*`).
- **Obserwujemy:** kod HTTP.

### M1.3 `test_protocol_version`

- **Rodzaj:** happy path + błąd
- **Stan początkowy:** Działający serwer.
- **Dane wejściowe:** Nagłówek `MCP-Protocol-Version`: `1999-01-01` i `2025-06-18`.
- **Kroki:** 1. Wysłać z nieobsługiwaną wersją. 2. Z obsługiwaną.
- **Oczekiwany wynik:** Nieobsługiwana → 400 i błąd JSON-RPC `-32600`; obsługiwana → 200.
- **Obserwujemy:** kod HTTP, `error.code`.

### M1.4 `test_invalid_json_and_batch`

- **Rodzaj:** błąd / złe dane
- **Stan początkowy:** Działający serwer.
- **Dane wejściowe:** Ciało `{nope`, bajty `\xff\xfe`, tablica (batch), `"x"`, `5`.
- **Kroki:** 1. Wysłać niepoprawny JSON i niepoprawne UTF-8. 2. Wysłać batch i wartości inne niż obiekt.
- **Oczekiwany wynik:** JSON/UTF-8 → 400 `-32700` (parse error); batch i nie-obiekt → 400 `-32600` (invalid request).
- **Obserwujemy:** kod HTTP, `error.code`.

### M1.5 `test_notification_gets_202`

- **Rodzaj:** happy path
- **Stan początkowy:** Działający serwer.
- **Dane wejściowe:** Powiadomienie `notifications/initialized` (bez `id`).
- **Kroki:** 1. Wysłać powiadomienie.
- **Oczekiwany wynik:** 202 i puste ciało.
- **Obserwujemy:** kod HTTP, długość ciała.

### M1.6 `test_other_methods_405`

- **Rodzaj:** błąd / złe dane
- **Stan początkowy:** Działający serwer.
- **Dane wejściowe:** `GET /mcp`, `DELETE /mcp`.
- **Kroki:** 1. Wysłać obie metody.
- **Oczekiwany wynik:** 405 i nagłówek `Allow: POST`.
- **Obserwujemy:** kod HTTP, nagłówek.

### M1.7 `test_origin_allowlist`

- **Rodzaj:** błąd / złe dane + happy path
- **Stan początkowy:** Osobna aplikacja z routerem MCP i listą dozwolonych źródeł `http://good.example`.
- **Dane wejściowe:** `ping` z `Origin` obcym, dozwolonym i bez `Origin`.
- **Kroki:** 1. Trzy żądania z różnym `Origin`.
- **Oczekiwany wynik:** Obce źródło → 403; dozwolone → 200; brak nagłówka (klient nie-przeglądarkowy) → 200.
- **Obserwujemy:** kody HTTP.

### M1.8 `test_loopback_url`

- **Rodzaj:** happy path
- **Stan początkowy:** Atrapa obiektu żądania z `server=("0.0.0.0", 1234)`.
- **Dane wejściowe:** Zmienna `MCP_LOOPBACK_URL` nieustawiona i ustawiona na `http://internal:9/`.
- **Kroki:** 1. Wyliczyć adres zwrotny bez zmiennej. 2. Ustawić zmienną i wyliczyć ponownie.
- **Oczekiwany wynik:** Domyślnie `http://127.0.0.1:1234`; ze zmienną `http://internal:9` (bez końcowego `/`).
- **Obserwujemy:** wartość zwrócona przez `_loopback_url`.

In [ ]:
!cd ../.. && python -m pytest tests/integration/mcp/test_mcp_transport.py -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

## M2. Sesja JSON-RPC po HTTP

**Plik:** `tests/integration/mcp/test_mcp_session.py`

**Integracja:** klient JSON-RPC ↔ `/mcp` ↔ `MCPServer` ↔ narzędzia ↔ REST ↔ dysk.

### M2.1 `test_full_session`

- **Rodzaj:** happy path
- **Stan początkowy:** Działający serwer, pusty vault.
- **Dane wejściowe:** `initialize`, `notifications/initialized`, `tools/list`, `tools/call create_note`, `ping`.
- **Kroki:** 1. Inicjalizacja. 2. Powiadomienie. 3. Lista narzędzi. 4. Utworzenie notatki. 5. `ping`.
- **Oczekiwany wynik:** Wersja protokołu `2025-06-18`, `serverInfo.name=notediscovery-mcp`; powiadomienie bez odpowiedzi; 18 narzędzi zgodnych z definicją; notatka `s.md` istnieje na dysku z treścią `hi`; `ping` → `{}`.
- **Obserwujemy:** odpowiedzi JSON-RPC, plik `s.md`.

### M2.2 `test_unknown_version_gets_newest`

- **Rodzaj:** błąd / złe dane
- **Stan początkowy:** Działający serwer.
- **Dane wejściowe:** `initialize` z `protocolVersion=1999-01-01`.
- **Kroki:** 1. Wysłać żądanie.
- **Oczekiwany wynik:** Serwer odpowiada najnowszą obsługiwaną wersją.
- **Obserwujemy:** `result.protocolVersion`.

### M2.3 `test_stateless_no_initialize_needed`

- **Rodzaj:** happy path
- **Stan początkowy:** Działający serwer, brak wcześniejszego `initialize`.
- **Dane wejściowe:** `tools/list`.
- **Kroki:** 1. Wywołać od razu.
- **Oczekiwany wynik:** Odpowiedź z listą narzędzi (transport HTTP jest bezstanowy).
- **Obserwujemy:** `result.tools`.

### M2.4 `test_errors`

- **Rodzaj:** błąd / złe dane
- **Stan początkowy:** Działający serwer.
- **Dane wejściowe:** Nieznana metoda (z `id` i bez), nieznane narzędzie `no_such_tool`.
- **Kroki:** 1. Wywołać `nope/method` z `id=7`. 2. Wywołać `nope/notification` bez `id`. 3. `tools/call` z nieznaną nazwą.
- **Oczekiwany wynik:** `-32601` (method not found); powiadomienie bez odpowiedzi; narzędzie → `isError=true` i „Unknown tool”.
- **Obserwujemy:** `error.code`, `isError`, tekst.

In [ ]:
!cd ../.. && python -m pytest tests/integration/mcp/test_mcp_session.py -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

## M3. Uwierzytelnianie `/mcp`

**Plik:** `tests/integration/mcp/test_mcp_auth.py`

**Integracja:** `/mcp` ↔ `require_auth` ↔ klucz API; wywołanie zwrotne narzędzia (loopback) do REST z kluczem; decyzja o montowaniu endpointu przy starcie.

### M3.1 `test_mcp_requires_key`

- **Rodzaj:** happy path + błąd
- **Stan początkowy:** Uwierzytelnianie włączone, klucz API ustawiony.
- **Dane wejściowe:** Nagłówki: brak, `X-API-Key: wrong`, poprawny `X-API-Key`, poprawny `Authorization: Bearer`.
- **Kroki:** 1. `POST /mcp` (ping) dla każdego wariantu.
- **Oczekiwany wynik:** Brak i zły klucz → 401 z JSON `{"detail":"Not authenticated"}` (nie przekierowanie na logowanie); poprawny klucz → 200.
- **Obserwujemy:** kod HTTP, treść JSON.

### M3.2 `test_loopback_call_carries_the_key`

- **Rodzaj:** happy path
- **Stan początkowy:** Uwierzytelnianie włączone; notatka `n.md` na dysku.
- **Dane wejściowe:** `tools/call list_notes` z poprawnym `X-API-Key`.
- **Kroki:** 1. Wywołać narzędzie przez `/mcp`. (Narzędzie wywołuje wewnętrznie `/api/notes`, które też wymaga klucza.)
- **Oczekiwany wynik:** Odpowiedź „Found 1 note(s)”, czyli klucz został przekazany w wywołaniu zwrotnym.
- **Obserwujemy:** tekst wyniku narzędzia.

### M3.3 `test_mount_decision_at_startup`

- **Rodzaj:** happy path + błąd
- **Stan początkowy:** Świeży proces Pythona dla każdej konfiguracji (decyzja zapada przy imporcie).
- **Dane wejściowe:** (1) auth włączone, brak klucza; (2) auth włączone, klucz `k`; (3) auth wyłączone, `MCP_ENABLED=false`; (4) auth wyłączone, MCP włączone.
- **Kroki:** 1. W podprocesie zaimportować aplikację i wysłać `ping` na `/mcp`.
- **Oczekiwany wynik:** Endpoint działa (200) tylko w (2) i (4); w (1) i (3) nie jest zamontowany.
- **Obserwujemy:** wynik w podprocesie (`True`/`False`).

In [ ]:
!cd ../.. && python -m pytest tests/integration/mcp/test_mcp_auth.py -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

## M4. Narzędzia do notatek

**Plik:** `tests/integration/mcp/test_mcp_note_tools.py`

**Integracja:** narzędzie MCP ↔ klient HTTP ↔ REST API ↔ system plików.

### M4.1 `test_create_get_append_move_delete`

- **Rodzaj:** happy path
- **Stan początkowy:** Pusty vault.
- **Dane wejściowe:** Notatka `zażółć/Łódź.md` (polskie znaki) o treści `# Zażółć\nbody`; dopiski `more`, `t`; nowa ścieżka `moved/n.md`.
- **Kroki:** 1. `create_note`. 2. `get_note`. 3. `append_to_note` (zwykły i z timestampem). 4. `move_note`. 5. `delete_note`.
- **Oczekiwany wynik:** Plik istnieje z treścią w UTF-8; `get_note` pokazuje treść i `Size:`; dopisek na końcu (`body\n\nmore`) i separator `---` z datą; po przeniesieniu stary plik znika, nowy istnieje; po usunięciu plik znika.
- **Obserwujemy:** pliki na dysku, tekst odpowiedzi narzędzi.

### M4.2 `test_create_folder`

- **Rodzaj:** happy path
- **Stan początkowy:** Pusty vault.
- **Dane wejściowe:** `create_folder` ze ścieżką `a/b`.
- **Kroki:** 1. Wywołać narzędzie.
- **Oczekiwany wynik:** Odpowiedź „Folder created”, katalog `a/b` na dysku.
- **Obserwujemy:** katalog w vaulcie.

### M4.3 `test_list_notes_and_pagination`

- **Rodzaj:** happy path
- **Stan początkowy:** Notatki `d/a.md`, `d/b.md`, `d/c.md`.
- **Dane wejściowe:** `list_notes` bez i z `limit=2`.
- **Kroki:** 1. Wywołać dwukrotnie.
- **Oczekiwany wynik:** „Found 3 note(s)” i folder `d/`; z limitem „(showing 2)” oraz informacja o kolejnych notatkach.
- **Obserwujemy:** tekst odpowiedzi.

### M4.4 `test_get_recent_notes`

- **Rodzaj:** happy path + graniczny
- **Stan początkowy:** Notatki: `new.md` (1 dzień), `newer.md` (0,5 dnia), `old.md` (30 dni) – daty modyfikacji ustawione na dysku.
- **Dane wejściowe:** `get_recent_notes` z `days=7` i `days=0`.
- **Kroki:** 1. Wywołać narzędzie dwukrotnie.
- **Oczekiwany wynik:** Dla 7 dni: bez `old.md`, `newer.md` przed `new.md`; dla 0 dni „No notes modified”.
- **Obserwujemy:** kolejność w tekście odpowiedzi.

### M4.5 `test_failures_are_reported_in_text`

- **Rodzaj:** błąd / złe dane
- **Stan początkowy:** Notatka `Note.md` o treści `x`.
- **Dane wejściowe:** `get_note` na `missing.md`, `create_note` na `note.md` (kolizja), `delete_note` na `missing.md`.
- **Kroki:** 1. Wywołać trzy narzędzia z błędnymi danymi.
- **Oczekiwany wynik:** Tekst odpowiedzi zawiera kod z REST („HTTP 404”, „HTTP 409”, „HTTP 404”); istniejąca notatka nietknięta. (Czy wynik jest oznaczony jako błąd, sprawdza test poniżej.)
- **Obserwujemy:** tekst odpowiedzi, plik `Note.md`.

### M4.6 `test_failures_set_is_error`

- **Rodzaj:** BUG (BR-3)
- **Stan początkowy:** Notatka `Note.md`.
- **Dane wejściowe:** Te same trzy błędne wywołania co wyżej.
- **Kroki:** 1. Wywołać narzędzia i odczytać flagę `isError`.
- **Oczekiwany wynik:** `isError=true` dla każdego wywołania zakończonego błędem REST (zgodnie z MCP, klient ma rozpoznać błąd po fladze).
- **Obserwujemy:** pole `isError`. **Rzeczywisty: brak flagi** (błąd jedynie w tekście). Test oznaczony `xfail`.

### M4.7 `test_bad_paths_rejected_before_any_request`

- **Rodzaj:** błąd / złe dane
- **Stan początkowy:** Pusty vault.
- **Dane wejściowe:** Ścieżki: `../evil.md` (create_note), `/etc/passwd` (get_note), `C:\\x.md` (delete_note), `a/../../x` (create_folder).
- **Kroki:** 1. Wywołać każde narzędzie z niebezpieczną ścieżką.
- **Oczekiwany wynik:** Każde zwraca tekst zaczynający się od „Error: path”; żądanie nie trafia do REST, na dysku nic nie powstaje.
- **Obserwujemy:** tekst wyniku, brak pliku `evil.md`.

In [ ]:
!cd ../.. && python -m pytest tests/integration/mcp/test_mcp_note_tools.py -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

## M5. Narzędzia tylko do odczytu

**Plik:** `tests/integration/mcp/test_mcp_read_tools.py`

**Integracja:** narzędzie MCP ↔ klient HTTP ↔ REST API ↔ indeks (wyszukiwanie, tagi, graf) i szablony.

### M5.1 `test_search`

- **Rodzaj:** happy path + błąd
- **Stan początkowy:** Notatki `a.md` (zawiera `findme`, link `[[b]]`, tag `alpha`) i `b.md`.
- **Dane wejściowe:** `search_notes` z `findme` i `zzz`.
- **Kroki:** 1. Wyszukać istniejącą frazę. 2. Wyszukać brakującą.
- **Oczekiwany wynik:** „Found 1 result(s)”, ścieżka `a.md` i fragment bez znaczników `<mark`; brak wyników → „No notes found matching 'zzz'”.
- **Obserwujemy:** tekst odpowiedzi.

### M5.2 `test_tags`

- **Rodzaj:** happy path + błąd
- **Stan początkowy:** Notatki `a.md`, `b.md` z tagiem `alpha`.
- **Dane wejściowe:** `list_tags`, `get_notes_by_tag` (z `limit=1` i z nieistniejącym tagiem).
- **Kroki:** 1. Lista tagów. 2. Notatki tagu. 3. Z limitem. 4. Tag `nope`.
- **Oczekiwany wynik:** `#alpha (2 notes)`; obie notatki; „more notes available” przy limicie; „No notes found with tag '#nope'”.
- **Obserwujemy:** tekst odpowiedzi.

### M5.3 `test_graph_and_backlinks`

- **Rodzaj:** happy path
- **Stan początkowy:** `a.md` linkuje do `b.md`.
- **Dane wejściowe:** `get_graph`, `get_backlinks` dla `b.md` i `a.md`.
- **Kroki:** 1. Pobrać graf. 2. Backlinki obu notatek.
- **Oczekiwany wynik:** „2 nodes, 1 connections”; `b.md` ma 1 backlink z `a.md`; `a.md` ma „No backlinks found”.
- **Obserwujemy:** tekst odpowiedzi.

### M5.4 `test_templates`

- **Rodzaj:** happy path + błąd
- **Stan początkowy:** Początkowo brak szablonów; potem dodany `_templates/Plan.md` (`# {{title}}`).
- **Dane wejściowe:** `list_templates`, `get_template` (`Plan`, `Nope`), `create_note_from_template`.
- **Kroki:** 1. Lista pusta. 2. Dodać szablon, lista, pobranie. 3. Pobrać nieistniejący. 4. Utworzyć notatkę `x/Roadmap.md`.
- **Oczekiwany wynik:** „No templates available”; potem `Plan` na liście i jego treść; `Nope` → „HTTP 404”; notatka `x/Roadmap.md` na dysku ma treść `# Roadmap`.
- **Obserwujemy:** tekst odpowiedzi, plik na dysku.

### M5.5 `test_health_and_config`

- **Rodzaj:** happy path
- **Stan początkowy:** Działający serwer.
- **Dane wejściowe:** `health_check`, `get_config`.
- **Kroki:** 1. Wywołać oba narzędzia.
- **Oczekiwany wynik:** „is healthy”; konfiguracja zawiera nagłówek „NoteDiscovery Server Configuration” i pole `name`.
- **Obserwujemy:** tekst odpowiedzi.

In [ ]:
!cd ../.. && python -m pytest tests/integration/mcp/test_mcp_read_tools.py -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

## M6. Transport stdio

**Plik:** `tests/integration/mcp/test_mcp_stdio.py`

**Integracja:** klient (proces `python -m mcp_server` przez stdin/stdout) ↔ serwer MCP ↔ żywy backend HTTP.

### M6.1 `test_session`

- **Rodzaj:** happy path
- **Stan początkowy:** Działający backend, pusty vault.
- **Dane wejściowe:** Linie JSON-RPC: `initialize`, powiadomienie, `tools/list`, `create_note`, `ping`.
- **Kroki:** 1. Uruchomić podproces i wysłać pięć linii.
- **Oczekiwany wynik:** Odpowiedzi o `id` 1, 2, 3, 4 (powiadomienie bez odpowiedzi); 18 narzędzi; notatka `s.md` o treści „via stdio” na dysku.
- **Obserwujemy:** stdout podprocesu, plik `s.md`.

### M6.2 `test_requires_initialize_and_survives_bad_line`

- **Rodzaj:** błąd / złe dane
- **Stan początkowy:** Działający backend.
- **Dane wejściowe:** `tools/list` przed `initialize`, uszkodzona linia `{not json`, `ping`.
- **Kroki:** 1. Wysłać trzy linie.
- **Oczekiwany wynik:** Pierwsza → błąd `-32002` (brak inicjalizacji); druga → `-32700`; trzecia → poprawny `{}`. Serwer nie kończy pracy po błędnej linii.
- **Obserwujemy:** stdout podprocesu.

### M6.3 `test_backend_down`

- **Rodzaj:** błąd / złe dane
- **Stan początkowy:** Brak backendu pod podanym adresem (port zwolniony).
- **Dane wejściowe:** `initialize`, potem `health_check`.
- **Kroki:** 1. Uruchomić klienta z adresem martwego serwera.
- **Oczekiwany wynik:** Tekst odpowiedzi: „not reachable” oraz „Connection error”.
- **Obserwujemy:** stdout podprocesu.

### M6.4 `test_wrong_api_key`

- **Rodzaj:** błąd / złe dane
- **Stan początkowy:** Backend z włączonym uwierzytelnianiem.
- **Dane wejściowe:** Klucz API `wrong`, narzędzie `list_notes`.
- **Kroki:** 1. Wywołać narzędzie.
- **Oczekiwany wynik:** Tekst odpowiedzi zawiera „HTTP 401”.
- **Obserwujemy:** stdout podprocesu.

### M6.5 `test_wrong_api_key_sets_is_error`

- **Rodzaj:** 🐞 BUG (BR-4)
- **Stan początkowy:** Backend z włączonym uwierzytelnianiem.
- **Dane wejściowe:** Klucz API `wrong`, narzędzie `list_notes`.
- **Kroki:** 1. Wywołać narzędzie i odczytać `isError`.
- **Oczekiwany wynik:** `isError=true` (błąd uwierzytelniania powinien być oznaczony jako błąd).
- **Obserwujemy:** pole `isError`. **Rzeczywisty: brak flagi**. Test oznaczony `xfail`.

In [ ]:
!cd ../.. && python -m pytest tests/integration/mcp/test_mcp_stdio.py -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

# Podsumowanie wyników

Porównanie wyniku oczekiwanego z rzeczywistym: oczekiwaliśmy, że wszystkie testy opisują poprawne działanie systemu. W praktyce wszystkie przechodzą poza 4, które ujawniły błędy aplikacji (XFAIL, patrz bug reporty).

| Sekcja | Plik | Przeszło | XFAIL (znane błędy) | Nie przeszło |
|---|---|---|---|---|
| A1 Notatki | `test_api_notes.py` | 5 | 1 | 0 |
| A2 Foldery i przenoszenie | `test_api_folders.py` | 6 | 1 | 0 |
| A3 Media (obrazy) | `test_api_media.py` | 6 | 0 | 0 |
| A4 Wyszukiwanie, tagi, graf, statystyki | `test_api_search_graph.py` | 5 | 0 | 0 |
| A5 Szablony | `test_api_templates.py` | 4 | 0 | 0 |
| A6 Udostępnianie notatek | `test_api_sharing.py` | 11 | 0 | 0 |
| A7 Eksport (HTML, ZIP) | `test_api_export.py` | 6 | 0 | 0 |
| A8 Uwierzytelnianie | `test_api_auth.py` | 9 | 0 | 0 |
| A9 System i wtyczki | `test_api_system_plugins.py` | 3 | 0 | 0 |
| M1 Transport HTTP `/mcp` | `test_mcp_transport.py` | 9 | 0 | 0 |
| M2 Sesja JSON-RPC po HTTP | `test_mcp_session.py` | 4 | 0 | 0 |
| M3 Uwierzytelnianie `/mcp` | `test_mcp_auth.py` | 9 | 0 | 0 |
| M4 Narzędzia do notatek | `test_mcp_note_tools.py` | 9 | 1 | 0 |
| M5 Narzędzia tylko do odczytu | `test_mcp_read_tools.py` | 5 | 0 | 0 |
| M6 Transport stdio | `test_mcp_stdio.py` | 4 | 1 | 0 |
| **Razem** | | **95** | **4** | **0** |

Uruchomienie całego zestawu naraz (liczby powinny zgadzać się z tabelą):

In [ ]:
!cd ../.. && python -m pytest tests/integration -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

# Bug reporty

Znaleźliśmy 4 rozbieżności między oczekiwanym a rzeczywistym zachowaniem aplikacji. Wszystkie dotyczą **kodów odpowiedzi/flag błędu**, żadna nie powoduje wycieku danych. BR-3 i BR-4 mają wspólną przyczynę.

## BR-1: Próba zapisu notatki poza katalogiem notatek kończy się HTTP 500

| Pole | Opis |
|---|---|
| Test, który to wykrywa | `tests/integration/api/test_api_notes.py::test_traversal_write_is_rejected_as_client_error` |
| Komponent z problemem | `backend/main.py`, endpoint `POST /api/notes/{note_path}` (zapis przez `save_note` z `backend/utils.py`) |
| Sposób komunikacji między komponentami | HTTP REST: klient → FastAPI → funkcja zapisu na dysku |
| Konfiguracja i dane testowe | Włączony zwykły tryb bez uwierzytelniania; pusty vault; obok vaultu plik `evil.md` (`keep`); żądanie `POST /api/notes/..%2Fevil.md` z `{"content":"pwn"}`. |
| Kroki | 1. Utworzyć plik `evil.md` obok katalogu notatek. 2. Wysłać `POST /api/notes/..%2Fevil.md`. |
| Oczekiwany wynik | Błąd klienta: 400 lub 403 (niedozwolona ścieżka). |
| Rzeczywisty wynik | 500 „Failed to save note”. Bezpieczeństwo zachowane (plik `evil.md` nietknięty), ale `save_note` zwraca `False` dla ścieżki spoza vaultu, a endpoint każde `False` zamienia na 500. |
| Priorytet | Niski (nie ma wycieku danych, tylko zły kod i szum w logach/monitoringu). |
| Proponowana poprawka | W endpoincie wywołać `validate_path_security` przed `save_note` i rzucić `HTTPException(403)`; albo rozróżnić w `save_note` „odmowę ścieżki” od „błędu zapisu” (wyjątek zamiast `False`) i mapować ją na 4xx. |

## BR-2: Usunięcie nieistniejącego folderu zwraca HTTP 500 zamiast 404

| Pole | Opis |
|---|---|
| Test, który to wykrywa | `tests/integration/api/test_api_folders.py::test_delete_missing_folder_is_404` |
| Komponent z problemem | `backend/main.py`, `delete_folder_endpoint` oraz `backend/utils.py::delete_folder` |
| Sposób komunikacji między komponentami | HTTP REST → funkcja pomocnicza (zwraca tylko `True`/`False`) |
| Konfiguracja i dane testowe | Pusty vault; `DELETE /api/folders/nope`. |
| Kroki | 1. Wysłać `DELETE /api/folders/nope`. |
| Oczekiwany wynik | 404 Not Found. |
| Rzeczywisty wynik | 500 „Failed to delete folder”. Funkcja `delete_folder` zwraca `False` dla brakującego folderu, dla ścieżki spoza vaultu, dla pliku zamiast folderu i dla wyjątku, a endpoint każdy `False` zamienia na 500. |
| Priorytet | Niski/średni (zły kod błędu utrudnia klientom API, np. ponowne usunięcie nie jest idempotentne). |
| Proponowana poprawka | Endpoint sprawdza istnienie folderu przed wywołaniem (404), a `delete_folder` zwraca przyczynę (wyjątki / enum) zamiast samego `False`. |

## BR-3: Błędy REST w narzędziach MCP (po HTTP) nie są oznaczane flagą `isError`

| Pole | Opis |
|---|---|
| Test, który to wykrywa | `tests/integration/mcp/test_mcp_note_tools.py::test_failures_set_is_error` |
| Komponent z problemem | `mcp_server/server.py` (`_tool_*`, `handle_call_tool`) |
| Sposób komunikacji między komponentami | JSON-RPC po HTTP (`/mcp`) → `MCPServer` → klient HTTP → REST. Błąd REST jest zamieniany na tekst („Failed to get note: HTTP 404…”). |
| Konfiguracja i dane testowe | Notatka `Note.md`; wywołania `get_note` (`missing.md`), `create_note` (`note.md`, kolizja), `delete_note` (`missing.md`). |
| Kroki | 1. `tools/call` z błędnymi argumentami. 2. Odczytać `result.isError`. |
| Oczekiwany wynik | `isError: true` (zgodnie ze specyfikacją MCP klient, w tym model AI, rozpoznaje niepowodzenie po fladze). |
| Rzeczywisty wynik | `result` zawiera tylko `content` z tekstem błędu, bez `isError`. Flaga jest ustawiana wyłącznie dla nieznanego narzędzia i wyjątku. To samo dotyczy lokalnej walidacji ścieżki (`Error: path …`). |
| Priorytet | Średni (agent AI może uznać nieudaną operację za udaną). |
| Proponowana poprawka | W `_tool_*` zwracać strukturę `(tekst, is_error)` albo wprowadzić wyjątek `ToolError`, który `handle_call_tool` zamienia na `isError: true`. |

## BR-4: Odrzucenie klucza API (401) przez stdio nie jest oznaczone `isError`

| Pole | Opis |
|---|---|
| Test, który to wykrywa | `tests/integration/mcp/test_mcp_stdio.py::test_wrong_api_key_sets_is_error` |
| Komponent z problemem | `mcp_server/server.py` (ta sama przyczyna co BR-3, widoczna w transporcie stdio) |
| Sposób komunikacji między komponentami | JSON-RPC po stdio → `MCPServer` → klient HTTP → backend z włączonym uwierzytelnianiem |
| Konfiguracja i dane testowe | Backend z uwierzytelnianiem; zmienne `NOTEDISCOVERY_URL`=adres backendu, `NOTEDISCOVERY_API_KEY=wrong`; narzędzie `list_notes`. |
| Kroki | 1. Uruchomić `python -m mcp_server`. 2. `initialize`, `tools/call list_notes`. 3. Odczytać `result.isError`. |
| Oczekiwany wynik | `isError: true`. |
| Rzeczywisty wynik | Tekst „Failed to list notes: HTTP 401: Not authenticated”, bez flagi `isError`. |
| Priorytet | Średni (jak BR-3; dodatkowo błąd konfiguracji klucza trudno automatycznie wykryć). |
| Proponowana poprawka | Jak w BR-3 (jedno miejsce poprawki w `mcp_server/server.py` naprawia oba transporty). |